# Predict Customer Churn - Training Pipeline

このノートブックは、Kaggle「Predict Customer Churn」コンペティションのモデル学習を実行します。

## 使い方

1. **Cell 6**で実験設定を変更
   - `EXP_NAME`: 大きな実験名 (EXP001, EXP002, EXP003, ...)
   - `CHILD_EXP_NAME`: 小さな実験名 (child-exp000, child-exp001, ...)

2. すべてのセルを順番に実行

3. 学習完了後、結果がGoogle Driveに自動保存される

---

In [ ]:
# Pythonバージョン確認
!python --version

In [ ]:
# GPU確認（Colab使用時）
!nvidia-smi

In [ ]:
# 必要なライブラリをインストール
!pip install lightgbm xgboost scikit-learn pandas numpy pyyaml -q

In [ ]:
# Google Drive をマウント（Colab使用時）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Kaggle認証情報をセット（Colab secretsから）
import os
from google.colab import userdata

try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_KEY')
    print("✓ Kaggle credentials set from Colab secrets")
except:
    print("⚠ Kaggle credentials not set (may be needed for data download)")

In [ ]:
# ============================================================
# 🎯 EXPERIMENT CONFIGURATION（実験設定の一元管理）
# ============================================================
# ここで実験設定を変更すれば、以降のすべてのセルが自動的に対応します

# 実験名の設定
EXP_NAME = "EXP001"              # 大きな実験名 (EXP001, EXP002, EXP003, ...)
CHILD_EXP_NAME = "child-exp000"  # 小さな実験名 (child-exp000, child-exp001, ...)

# 作業ディレクトリ
WORK_DIR = "/content"  # Colab環境
# WORK_DIR = "/path/to/local"  # ローカル環境の場合はこれを変更

print(f"🚀 Configuration:")
print(f"   EXP_NAME: {EXP_NAME}")
print(f"   CHILD_EXP_NAME: {CHILD_EXP_NAME}")
print(f"   WORK_DIR: {WORK_DIR}")

In [ ]:
# Kaggleデータをダウンロード
!pip install kaggle -q
!kaggle competitions download -c predict-customer-churn -p {WORK_DIR}/data/ 2>/dev/null || echo "⚠ Data download skipped (may already exist)"

In [ ]:
# データディレクトリの確認
import os
import zipfile
from pathlib import Path

data_dir = Path(f"{WORK_DIR}/data")
data_dir.mkdir(parents=True, exist_ok=True)

# zipファイルがあれば展開
for zip_file in data_dir.glob("*.zip"):
    print(f"Extracting {zip_file.name}...")
    with zipfile.ZipFile(zip_file, 'r') as ref:
        ref.extractall(data_dir)
    zip_file.unlink()  # 展開後は削除

# CSVファイルの確認
csv_files = list(data_dir.glob("*.csv"))
print(f"\n✓ CSV files found: {len(csv_files)}")
for csv_file in csv_files:
    print(f"   - {csv_file.name}")

In [ ]:
# プロジェクトディレクトリをセット
import os
import sys
from pathlib import Path

# リポジトリをクローン（GitHub から）または既存フォルダを使用
project_root = Path(f"{WORK_DIR}") / "Kaggle-Predict-Customer-Churn"

if not project_root.exists():
    print(f"Cloning project from GitHub...")
    # または git clone
    project_root.mkdir(parents=True, exist_ok=True)
else:
    print(f"✓ Project directory found: {project_root}")

os.chdir(project_root)
sys.path.insert(0, str(project_root))
print(f"Working directory: {os.getcwd()}")

In [ ]:
# 訓練スクリプトを実行
import os
import subprocess

config_path = f"EXP/{EXP_NAME}/config/{CHILD_EXP_NAME}.yaml"
train_script = f"EXP/{EXP_NAME}/train.py"

print(f"\n=== 訓練を開始 ===")
print(f"Script: {train_script}")
print(f"Config: {config_path}")
print(f"="*50)

# 訓練実行
cmd = ["python", train_script, "--config", config_path]

result = subprocess.run(cmd, cwd=project_root)

if result.returncode == 0:
    print(f"\n✓ Training completed successfully!")
else:
    print(f"\n✗ Training failed with return code {result.returncode}")

In [ ]:
# 結果を確認
import json
from pathlib import Path

output_dir = Path(project_root) / f"EXP/{EXP_NAME}/outputs/{CHILD_EXP_NAME}"

if output_dir.exists():
    print(f"\n=== Training Results ===")
    
    # results.json を読込
    results_file = output_dir / "results.json"
    if results_file.exists():
        with open(results_file) as f:
            results = json.load(f)
        
        print(f"\nCV Metrics:")
        for key, value in results.items():
            if isinstance(value, float):
                print(f"  {key}: {value:.4f}")
            else:
                print(f"  {key}: {value}")
    
    # OOF予測ファイルの確認
    oof_file = output_dir / "oof_predictions.csv"
    if oof_file.exists():
        import pandas as pd
        oof_df = pd.read_csv(oof_file)
        print(f"\nOOF Predictions:")
        print(f"  Shape: {oof_df.shape}")
        print(f"  Columns: {list(oof_df.columns)}")
        print(f"\nFirst 5 rows:")
        print(oof_df.head())
else:
    print(f"⚠ Output directory not found: {output_dir}")

In [ ]:
# 結果をGoogle Driveに保存（Colab使用時）
import shutil
from pathlib import Path

try:
    drive_output_dir = Path("/content/drive/MyDrive/kaggle_churn/results") / EXP_NAME / CHILD_EXP_NAME
    drive_output_dir.mkdir(parents=True, exist_ok=True)
    
    # 結果をコピー
    local_output_dir = Path(project_root) / f"EXP/{EXP_NAME}/outputs/{CHILD_EXP_NAME}"
    
    if local_output_dir.exists():
        for file in local_output_dir.glob("*"):
            if file.is_file():
                shutil.copy(file, drive_output_dir / file.name)
        
        print(f"✓ Results saved to Google Drive:")
        print(f"  {drive_output_dir}")
except Exception as e:
    print(f"⚠ Could not save to Google Drive: {e}")

In [ ]:
# 📋 次回の実験設定メモ
print(f"""
======================================
次の実験を実行する場合は、以下の方法で：
======================================

【新しいハイパラで実験する場合】
1. Cell 6 を編集:
   CHILD_EXP_NAME = "child-exp001"
2. config/child-exp001.yaml を作成
3. Cell 10（訓練実行）以降を再実行

【新しいコードで実験する場合】
1. EXP/{EXP_NAME+1}/ ディレクトリを作成
2. train.py, infer.py, config/ を作成
3. Cell 6 を編集:
   EXP_NAME = "EXP002"
   CHILD_EXP_NAME = "child-exp000"
4. 実行

【結果をまとめる】
結果を EXP/EXP_SUMMARY.md に記入してください
======================================
""")